# Health- vs. Disease-Associated Species

The real GMWI2 (and its predecessor, the original GMHI) is built around a
simple idea before any machine learning enters the picture: some species
show up disproportionately often in healthy people across many studies
("health-prevalent"), and others show up disproportionately often in
people with a disease ("health-scarce"). This notebook hand-computes a
simplified version of that idea. **This is illustrative, not the real
trained model** — the actual GMWI2 species sets and weights come from
statistical analysis of 8,069 real samples, not a hand-picked list.

## 1. Setup — reuse notebook 03's presence/absence matrix

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("toy_species_abundance.csv")

health_prevalent = ["Faecalibacterium_prausnitzii", "Akkermansia_muciniphila",
                    "Bifidobacterium_adolescentis", "Eubacterium_rectale",
                    "Roseburia_intestinalis", "Alistipes_putredinis"]
health_scarce = ["Escherichia_coli", "Klebsiella_pneumoniae", "Enterococcus_faecalis",
                 "Clostridium_bolteae", "Fusobacterium_nucleatum", "Veillonella_parvula"]

presence = (df[health_prevalent + health_scarce] > 0).astype(int)
presence.insert(0, "health_status", df["health_status"])
presence.head(4)

## 2. A simplified log-ratio score

The real GMWI is (in spirit) a log-ratio of collective health-prevalent vs.
health-scarce signal. A minimal version:

$$\text{score} = \ln\left(\frac{1 + \sum \text{prevalent present}}{1 + \sum \text{scarce present}}\right)$$

The `1 +` on each side avoids taking `log(0)` when a sample has zero
species from one group — a standard trick (you saw why `log(0)` is a
problem in the other course's diversity-metrics notebook).

In [ ]:
def simple_health_score(row):
    n_prevalent = row[health_prevalent].sum()
    n_scarce = row[health_scarce].sum()
    return np.log((1 + n_prevalent) / (1 + n_scarce))

presence["score"] = presence.apply(simple_health_score, axis=1)
presence.groupby("health_status")["score"].describe()[["mean", "std", "min", "max"]].round(3)

### 🔧 YOUR TURN #1
Positive scores lean "healthy," negative scores lean "non_healthy" (same
convention the real GMWI2 uses). Count how many samples your simplified
score gets "wrong" — a `healthy` sample with a negative score, or a
`non_healthy` sample with a positive score:

```python
wrong = ((presence.health_status == "healthy") & (presence.score < 0)) | \
        ((presence.health_status == "non_healthy") & (presence.score > 0))
wrong.sum()
```
Run it, then compare that count to the total sample count (80). Does a
score this simple do better than random guessing?

In [ ]:
# Your code here

### EXPLAIN #1
*This score only counts how many health-prevalent/health-scarce species
are present — it ignores everything else in the sample (the other course's
Faecalibacterium, Bacteroides, Prevotella data isn't used at all here).
Why might a real model want to use ALL detected species, not just a
hand-picked list of 12?*

> your answer here

## Done — a real idea, an oversimplified formula

The direction of this idea is genuinely how GMHI/GMWI2 started. What makes
GMWI2 better than a fixed formula like this one is exactly what notebook
05 covers: learning the weights (which species matter, and how much) from
data, instead of guessing them.

**One thing worth noticing when you get to notebook 05:** this notebook's
score may look *more* accurate than notebook 05's trained classifier. That
is not a contradiction — it is a warning. You checked this score against
the exact same 80 samples used to pick which species counted as
"health-prevalent" and "health-scarce" in the first place. Notebook 05
checks its model with cross-validation: every prediction is made by a
version of the model that never saw that sample during training. An
easier test will always look better than a harder, more honest one — the
same lesson as "don't trust a score you trained on" in notebook 05
itself.

**Next:** `05_training_a_health_classifier.ipynb`.